# IADD Traffic Detection - YOLOv11 (Run 4, eval v5)

Same trained model as v4 (yolo11s, imgsz 1280, leakage-free by-video split).
Only the evaluation changed:
- mAP is reported at the default threshold, where it is averaged over all
  confidences and therefore threshold-independent - the honest headline number.
- The operating confidence for the confusion matrix and the demo is taken from
  Run-4's own val F1 curve (found automatically), not hard-coded. Tuning on val and
  reporting on test avoids tuning on the test set.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive

The run folder lives on Drive so a limited free session can be continued on another
Google account: share the `iadd_runs` folder and the dataset zip, add a shortcut to
that account's My Drive root, then run every cell - training resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the dataset into Colab

Colab's local disk is wiped on reconnect, so re-run this cell after any reconnect.

In [ ]:
!rm -rf /content/iadd_subset_v3
!unzip -q "/content/drive/MyDrive/iadd_subset_v3.zip" -d /content
!ls /content/iadd_subset_v3

## 5. Train - fresh, or auto-resume from Drive

If a checkpoint exists on Drive it resumes from the last epoch; otherwise it starts a
fresh Run 4. This is what makes multi-account training work: switch account, re-run,
and it continues where the previous session stopped.

In [ ]:
from ultralytics import YOLO
import os

MODEL    = 'yolo11s.pt'
IMGSZ    = 1280
BATCH    = 12         # fits imgsz 1280 on a 15 GB T4
EPOCHS   = 150
PATIENCE = 20

DRIVE_PROJECT = '/content/drive/MyDrive/iadd_runs'
RUN_NAME      = 'run4'
DATA          = '/content/iadd_subset_v3/data.yaml'
CKPT          = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT)
    results = model.train(resume=True)
else:
    print('no checkpoint -> fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH)
    model = YOLO(MODEL)
    results = model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=DRIVE_PROJECT, name=RUN_NAME, exist_ok=True, plots=True,
    )

## 6. Progress so far (optional)

Prints the per-epoch metrics table - useful after a reconnect to see the trend.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/iadd_runs/run4/results.csv')
df.columns = [c.strip() for c in df.columns]
print('epochs done:', len(df))
df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
    'metrics/precision(B)', 'metrics/recall(B)']].tail(10)

## 7. Validate, then pick the operating confidence

First a standard validation of the best checkpoint (COCO-standard mAP over all
confidences + per-class table + P/R/F1/PR curves). Then read the confidence that
maximises mean F1 straight from the val F1 curve, and render a clean confusion
matrix at that point in `run4_val_tuned`.

In [ ]:
from ultralytics import YOLO
import numpy as np

DATA = '/content/iadd_subset_v3/data.yaml'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run4/weights/best.pt')

# Honest metrics: mAP is averaged over all thresholds, so it is not tied to any conf.
mv = best.val(data=DATA, split='val', plots=True,
              project=RUNS, name='run4_val', exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f' % (mv.box.map50, mv.box.map))

# Operating point = confidence with the highest mean F1, read from the val F1 curve.
CONF = 0.25
for x, y, _, ylabel in mv.curves_results:
    if ylabel == 'F1':
        px = np.asarray(x)
        y = np.asarray(y)
        f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('F1-optimal confidence (from val) =', CONF)

# Clean confusion matrix + per-class table at that operating point.
best.val(data=DATA, split='val', conf=CONF, plots=True,
         project=RUNS, name='run4_val_tuned', exist_ok=True)
print('clean val plots -> run4_val_tuned  (conf=%.3f)' % CONF)

## 8. Test - honest mAP, and a comparable confusion matrix

The test videos were never seen in training (leakage-free split), so this is the
number to report. The second call uses the same operating confidence, so its
confusion matrix is directly comparable to `run3_test_tuned`.

In [ ]:
mt = best.val(data=DATA, split='test', plots=True,
              project=RUNS, name='run4_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f' % (mt.box.map50, mt.box.map))

best.val(data=DATA, split='test', conf=CONF, plots=True,
         project=RUNS, name='run4_test_tuned', exist_ok=True)
print('clean test plots -> run4_test_tuned  (conf=%.3f)' % CONF)

## 9. Demo predictions - saved to run4/demo

Runs the model on a few test images at the operating confidence and writes the
annotated images to a folder on Drive. Nothing is displayed inline.

In [ ]:
import glob

imgs = sorted(glob.glob('/content/iadd_subset_v3/images/test/*.jpg'))[:16]
best.predict(imgs, conf=CONF, save=True,
             project=f'{RUNS}/run4', name='demo', exist_ok=True)
print('demo images saved -> iadd_runs/run4/demo  (conf=%.3f)' % CONF)

---
Results on Drive under `iadd_runs/`:
- `run4/` - training, weights (`best.pt`, `last.pt`), and `demo/`
- `run4_val/`, `run4_test/` - honest metrics + curves (default threshold)
- `run4_val_tuned/`, `run4_test_tuned/` - clean confusion matrix at the F1-optimal conf